In [1]:
import sys
sys.path.append("../")

import folium
from folium.plugins import TimestampedGeoJson
import pandas as pd
import geopandas as gpd

import src.paths as PATHS
import src.constants as CONST

In [2]:
sam_data = PATHS.DATA_DIR / "sam" / "sam_processed.gpkg"

sam_dict = {}
for _, layer_data in gpd.list_layers(sam_data).iterrows():
    sam_dict[layer_data["name"]] = gpd.read_file(sam_data, layer=layer_data["name"])

In [3]:
sam_dict.keys()

dict_keys(['20250915', 'Ingetekende _Erosie_Rijntakken_2024', '20250915_Pt'])

In [4]:
sam_dict["Ingetekende _Erosie_Rijntakken_2024"].head()

,Type_Erosie,Uiterwaarde,Opmerking,geometry
0,PVR NG,Domswaard,Stabiel onstaan door kleiwinning,"LINESTRING (602673.46 6796238.496, 602712.948 ..."
1,PVR NG,Gamerensche waarden,NaN,"LINESTRING (579112.902 6764909.001, 579122.973..."
2,OE,Brakelsche benedenwaarden,NaN,"LINESTRING (561238.29 6766751.513, 561198.542 ..."
3,PVR NG,Graafsche waard,Middel,"LINESTRING (556412.209 6796286.697, 556438.949..."
4,PVR NG,Graafsche waard,Middel,"LINESTRING (556794.307 6796294.437, 556818.232..."


In [5]:
sam_dict["20250915_Pt"]["observation_date"] = pd.to_datetime(sam_dict["20250915_Pt"]["observation_date"])

sam_dict["20250915_Pt"].sample(5)

,_vertex_number,observation_id,patch_id,observation_date,water_height_m,rotation_angle_deg,source_image_filename,mask_image_filename,soil_image_filename,Year,geometry
7022,10312,25917,11338,2017-08-28,70.0,0.0,None,None,None,2017,POINT (6.11142 52.28902)
45480,27666,34707,12436,2024-01-01,70.0,0.0,None,None,None,2024,POINT (5.22924 51.74098)
68619,1450,34600,12423,2020-05-06,70.0,0.0,None,None,None,2020,POINT (5.26698 51.74011)
918,10359,4479,632,2016-05-12,70.0,0.0,None,None,None,2016,POINT (5.01003 51.81946)
118358,1816,4727,663,2017-03-16,70.0,0.0,None,None,None,2017,POINT (5.04551 51.81186)


In [ ]:
luke_data = PATHS.DATA_DIR / "phase1_2025-08-14_v1.gpkg"

luke_dict = {}
for _, layer_data in gpd.list_layers(luke_data).iterrows():
    luke_dict[layer_data["name"]] = gpd.read_file(luke_data, layer=layer_data["name"])

In [ ]:
luke_dict.keys()

In [ ]:
luke_dict["punten_oever"].sample(5)

In [ ]:
luke_dict["punten_oever"]["observation_date"] = pd.to_datetime(luke_dict["punten_oever"]["dtm_date"].map(lambda x: f"{x}-06-01"))

In [ ]:
mapa = folium.Map(location=[CONST.CENTRE_NL_LAT, CONST.CENTRE_NL_LON], zoom_start=CONST.DEFAULT_NL_ZOOM, control_scale=True)

# add scope
fg_scope = folium.FeatureGroup(name="scope regions", show=True).add_to(mapa)
folium.GeoJson(luke_dict["vlakken_scope"]).add_to(fg_scope)

# add SAM data
sam_layer = folium.FeatureGroup(name="SAM detections", show=True).add_to(mapa)

sam_dict["20250915_Pt"]["origin"] = "SAM"
luke_dict["punten_oever"]["origin"] = "HM"

all_points = pd.concat([sam_dict["20250915_Pt"].to_crs(epsg=CONST.EPSG_WGS84), luke_dict["punten_oever"].to_crs(epsg=CONST.EPSG_WGS84)], ignore_index=True)
all_points = all_points.loc[all_points.within(luke_dict["vlakken_scope"].to_crs(epsg=CONST.EPSG_WGS84)["geometry"].union_all())]

geojson_features = []
for _, row in all_points.iterrows():
    # TODO: lines are not rendered even though this looks exactly like the example in the docs where lines are rendered?!
    point = row["geometry"].__geo_interface__
    # line["coordinates"] = [list(a) for a in line['coordinates']]  # properly deal with a LineString
    geojson_features.append({
        "type": "Feature",
        "geometry": point,
        "properties": { 
            "times": [row["observation_date"].strftime("%Y-%m-%dT%H:%M:%S")],
            "icon": "circle",
            "style": {"color": "orange" if row["origin"] == "SAM" else "blue", "fillOpacity": 0.6, "radius_m": 2, "fillColor": "orange" if row["origin"] == "SAM" else "blue"},
        },
    })

geojson_data = {"type": "FeatureCollection", "features": geojson_features}
TimestampedGeoJson(
    geojson_data,
    period="P1Y",
    duration="P11M",
    transition_time=200,  # Milliseconds between frames
    loop=False,            # Loop animation
    auto_play=False,      # Start playing automatically
    loop_button=True,
).add_to(mapa)


folium.LayerControl().add_to(mapa)

mapa.save("sam_vs_height.html")